In [ ]:
from googleapiclient.discovery import build
import pandas as pd

# API key 
API_KEY = "XYZ"
youtube = build("youtube", "v3", developerKey=API_KEY)

# ------------------------
# Function to search channels
# ------------------------
def search_channels(query, max_results=50, pages=5):
    results = []
    next_page_token = None

    for _ in range(pages):
        request = youtube.search().list(
            q=query,
            type="channel",
            part="snippet",
            regionCode="GB",   # UK region
            maxResults=max_results,
            pageToken=next_page_token
        )
        response = request.execute()
        for item in response["items"]:
            results.append({
                "Channel_Name": item["snippet"]["title"],
                "Channel_ID": item["id"]["channelId"],
                "Channel_Link": f"https://www.youtube.com/channel/{item['id']['channelId']}",
                "Description": item["snippet"]["description"]
            })
        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break
    return results

# ------------------------
# Function to get channel details
# ------------------------
def get_channel_details(channel_ids):
    details = []
    for i in range(0, len(channel_ids), 50):  # 50 IDs per request
        request = youtube.channels().list(
            part="statistics,snippet",
            id=",".join(channel_ids[i:i+50])
        )
        response = request.execute()
        for item in response["items"]:
            details.append({
                "Channel_ID": item["id"],
                "Subscribers": int(item["statistics"].get("subscriberCount", 0)),
                "Video_Count": int(item["statistics"].get("videoCount", 0)),
                "Country": item["snippet"].get("country")
            })
    return details

# ------------------------
# Queries (Tech + Gaming Review)
# ------------------------
queries = [
    # Tech Review
    "UK tech review", "UK smartphone review", "UK laptop review",
    "UK gadget review", "UK unboxing", "UK electronics review",
    "UK PC hardware review", "UK camera review", "UK headphone review",
    "UK smartwatch review",

    # Gaming Review
    "UK gaming review", "UK gaming setup", "UK gaming laptop review",
    "UK console review", "UK PC build review", "UK gaming monitor review",
    "UK gaming equipment review"
]

all_channels = []
for q in queries:
    all_channels.extend(search_channels(q))

# DataFrame banake duplicates hatao
df = pd.DataFrame(all_channels).drop_duplicates(subset=["Channel_ID"])

# Channel stats
details = get_channel_details(df["Channel_ID"].tolist())
df_details = pd.DataFrame(details)

# Merge
final_df = df.merge(df_details, on="Channel_ID", how="left")

# ------------------------
# Filter: UK + Relevant Keywords
# ------------------------
final_df = final_df[final_df["Country"] == "GB"]

# Keywords filter
keywords = ["review", "tech", "unboxing", "gadget", "electronics",
            "gaming", "pc build", "console", "setup", "equipment"]
def is_relevant_channel(row):
    text = (str(row["Channel_Name"]) + " " + str(row["Description"])).lower()
    return any(kw in text for kw in keywords)

final_df = final_df[final_df.apply(is_relevant_channel, axis=1)]

# Sort by Subscribers (optional, but better for ranking)
final_df = final_df.sort_values(by="Subscribers", ascending=False)

# Top 500 channels
final_df = final_df.head(500)

# ------------------------
# Save to Excel
# ------------------------
save_path = "YOUR_PATH"
final_df.to_excel(save_path, index=False)

print(f"Saved: {save_path}")